# Topic 22f - Writing the wrangling report

The main lesson showed one complete worked wrangling report. This notebook gives you a checklist for judging one, a bad-versus-good comparison, and practice writing your own paragraph about a Topic 20 decision the worked example did not cover.

Work through it after reading section 7 of the [main lesson](topic22_exploratory_data_analysis.ipynb).

In [1]:
# Run this cell as it is.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "exploratory_data_analysis" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "air_quality_readings.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find air_quality_readings.csv. Looked in: " +
        ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e1e0d9"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.edgecolor"] = "#c3c2b7"
plt.rcParams["axes.axisbelow"] = True

# The course palette (Topic 21) - fixed hue order, never cycled, so a category
# keeps the same colour in every chart it appears in.
PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"
INK = "#0b0b0b"
STATION_TYPE_COLOR = {
    "Roadside": PALETTE["red"], "Industrial": PALETTE["orange"],
    "Residential": PALETTE["aqua"], "Background": PALETTE["blue"],
}
STATION_TYPE_ORDER = ["Roadside", "Industrial", "Residential", "Background"]

WHO_ANNUAL_GUIDELINE = 5.0   # ug/m3, PM2.5 annual mean
WHO_24H_GUIDELINE = 15.0     # ug/m3, PM2.5 24-hour mean

print("pandas version:   ", pd.__version__)
print("data folder:      ", DATA_DIR)
print("files:            ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version:    3.0.6
data folder:       /home/johnny/Projects/itfy-da/exploratory_data_analysis/data
files:             ['air_quality_readings.csv', 'air_quality_readings_assignment.csv', 'air_quality_stations.csv', 'air_quality_stations_assignment.csv', 'customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


In [2]:
# The retail "2023 Trading Review" table from Topic 20 - used for the two
# report-writing sections later in this notebook, since a wrangling report and
# an insights report have to be about work you actually did.
def clean_region(series):
    return (series.str.strip().str.replace(r"\s+", " ", regex=True).str.title())


def build_analysis_table(orders, customers, products, regions, returns):
    expected_rows = len(orders)
    customers = customers.copy()
    customers["region"] = clean_region(customers["region"])
    products = products.drop_duplicates(subset="product_id", keep="first")
    customers = customers.merge(regions, on="region", how="left", validate="m:1")
    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))
    returned_ids = set(returns["order_id"])
    table["was_returned"] = table["order_id"].isin(returned_ids)
    table["order_date"] = pd.to_datetime(table["order_date"])
    table["month"] = table["order_date"].dt.strftime("%b")
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    if len(table) != expected_rows:
        raise ValueError("Row count changed during wrangling.")
    return table.sort_values("order_date").reset_index(drop=True)


orders = pd.read_csv(DATA_DIR / "orders.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")

analysis = build_analysis_table(orders, customers, products, regions, returns)
print(f"Retail analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")

Retail analysis table: 1200 rows x 23 columns


---

## A checklist for a wrangling report

A wrangling report earns its keep if, and only if, a reader who never opens the notebook could still answer three questions from it alone:

- [ ] What problems were in the raw data?
- [ ] What was done about each one?
- [ ] What should the reader still not fully trust?

And it fails on tone if it does any of these:
- [ ] Shows code, a column name, or a library function name
- [ ] Uses the word "just" to describe a decision with a real trade-off ("I just dropped the bad rows")
- [ ] States a fix without a number attached (how many rows, what percentage)

---

## Bad versus good, same issue

Here is the orphan-order issue from the main lesson's worked example, written badly first.

> **Bad:** "I ran a left join on customer_id and found some nulls in the region column, which I kept in the DataFrame after checking `isna().sum()`. There were 27 of them."

This is accurate and still a bad wrangling report. It assumes the reader knows what a left join is, states a number with no context for whether 27 is a lot, and never says what a reader should actually do about it.

> **Good:** "27 of the distributor's 1,200 orders (2.3%) came from a customer who is not in the customer list - likely a data-entry error when the order was logged. These orders are kept, because removing them would understate 2023 revenue by GHS 9,090.87, but they cannot be assigned to a region. Anyone using regional totals should know that 2.3% of revenue sits outside every region's figure."

Same fact, same number, completely different report. The second version never mentions pandas and is more useful to a non-technical reader than the first.

### Try It 7 - write your own paragraph

Topic 20 also found that `targets_wide.csv` needed melting from wide to long format before it could be joined - a reshaping decision, not a data-quality fix, but still something a wrangling report should mention if it changed what a reader can trust about the targets. Write one paragraph (3-5 sentences, no code, no column names) explaining this decision to someone who has never seen the file. Then check it against the checklist above.

In [3]:
my_paragraph = """
YOUR PARAGRAPH HERE
"""

word_count = len(my_paragraph.split())
print(f"Word count: {word_count}")
print(f"Contains no code-like text (no underscores, no parentheses): "
      f"{'_' not in my_paragraph and '(' not in my_paragraph}")

Word count: 3
Contains no code-like text (no underscores, no parentheses): True


---
## Where this fits

- [Back to the main lesson](topic22_exploratory_data_analysis.ipynb)
- Previous: [22e - From patterns to insights](topic22e_patterns_to_insights.ipynb)
- Next: [22g - Writing the act report](topic22g_writing_the_act_report.ipynb)